[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module7/07-huggingface.ipynb)

# Hugging Face Transformers Library
**Module 7 — Lesson 7 | Estimated time: 30 minutes**

> 💡 Enable GPU: Runtime → Change runtime type → GPU

## Learning Objectives
By the end of this notebook you will be able to:
- Use `pipeline()` for plug-and-play NLP tasks
- Tokenise text with `AutoTokenizer` (encode, decode, batch, pad, truncate)
- Understand special tokens, attention masks, and padding
- Load `AutoModel` and task-specific model variants
- Browse and use the Hugging Face Model Hub
- Save and load models locally

In [ ]:
!pip install -q transformers datasets

In [ ]:
import torch
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
)
print('Transformers imported successfully.')
device = 0 if torch.cuda.is_available() else -1  # pipeline device arg
print('Using device index:', device)

## 1. Pipelines — Instant NLP

`pipeline()` wraps tokenisation, model inference, and post-processing into a single callable. All models below are small so they download quickly in Colab.

In [ ]:
# 1a. Sentiment Analysis (DistilBERT-based)
sentiment = pipeline('sentiment-analysis',
                     model='distilbert-base-uncased-finetuned-sst-2-english',
                     device=device)
results = sentiment([
    'PyTorch makes deep learning fun!',
    'Debugging NaN losses is incredibly frustrating.',
    'The model trained in under 5 minutes — great!',
])
for text, res in zip(['fun', 'frustrating', 'great'], results):
    print(f'  [{text}] -> {res["label"]} ({res["score"]:.3f})')

In [ ]:
# 1b. Text Generation (GPT-2 small)
gen = pipeline('text-generation', model='gpt2', device=device)
out = gen('Deep learning is', max_new_tokens=40, num_return_sequences=2, do_sample=True, temperature=0.9)
print('=== Generated Texts ===')
for i, g in enumerate(out):
    print(f'[{i+1}]', g['generated_text'][:120])

In [ ]:
# 1c. Named Entity Recognition
ner = pipeline('ner', model='elastic/distilbert-base-uncased-finetuned-conll03-english',
               aggregation_strategy='simple', device=device)
entities = ner('Yann LeCun works at Meta AI in New York and won the Turing Award in 2018.')
for e in entities:
    print(f'  {e["word"]:20s} {e["entity_group"]:6s} ({e["score"]:.3f})')

In [ ]:
# 1d. Question Answering
qa = pipeline('question-answering',
              model='distilbert-base-cased-distilled-squad', device=device)
context = (
    'PyTorch is an open-source machine learning framework developed by Meta AI. '
    'It was released in 2016 and is written primarily in Python and C++.'
)
questions = [
    'Who developed PyTorch?',
    'When was PyTorch released?',
    'What language is PyTorch written in?',
]
for q in questions:
    ans = qa(question=q, context=context)
    print(f'Q: {q}\nA: {ans["answer"]} ({ans["score"]:.3f})\n')

In [ ]:
# 1e. Summarisation (T5-small)
summariser = pipeline('summarization', model='t5-small', device=device)
long_text = (
    'Transformers are a type of neural network architecture that has revolutionised '
    'natural language processing. Originally proposed in the 2017 paper \'Attention Is All You Need\' '
    'by Vaswani et al., transformers rely entirely on self-attention mechanisms rather than recurrent '
    'or convolutional layers. They have since become the dominant architecture for tasks including '
    'machine translation, text generation, question answering, and image recognition.'
)
summary = summariser(long_text, max_length=50, min_length=20, do_sample=False)
print('Summary:', summary[0]['summary_text'])

## 2. AutoTokenizer — Deep Dive

Tokenisers convert raw text to token IDs that models can process. Understanding tokenisation is crucial for debugging and fine-tuning.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

# Basic encode / decode
text = 'Hello, Hugging Face transformers!'
ids  = tokenizer.encode(text)
print('Token IDs:   ', ids)
print('Decoded:     ', tokenizer.decode(ids))
print('Token list:  ', tokenizer.convert_ids_to_tokens(ids))

In [ ]:
# Special tokens
print('Vocab size:  ', tokenizer.vocab_size)
print('[CLS] id:    ', tokenizer.cls_token_id)
print('[SEP] id:    ', tokenizer.sep_token_id)
print('[PAD] id:    ', tokenizer.pad_token_id)
print('[MASK] id:   ', tokenizer.mask_token_id)

# Batch encoding with padding and truncation
texts = [
    'Short sentence.',
    'This is a much longer sentence that will require more tokens to encode properly.',
    'Medium length text here for comparison.',
]
encoded = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=24,
    return_tensors='pt'
)
print('\nBatch encoded:')
print('input_ids shape:      ', encoded['input_ids'].shape)
print('attention_mask shape: ', encoded['attention_mask'].shape)
print('input_ids[1] (long):  ', encoded['input_ids'][1].tolist())
print('attention_mask[1]:    ', encoded['attention_mask'][1].tolist())

## 3. AutoModel and AutoModelForSequenceClassification

In [ ]:
# AutoModel — base model, returns hidden states
model_base = AutoModel.from_pretrained('distilbert-base-uncased')
with torch.no_grad():
    outputs = model_base(**encoded)
hidden = outputs.last_hidden_state
print('Base model hidden states:', hidden.shape)  # (batch, seq, 768)
print('CLS token representation:', hidden[:, 0, :].shape)

# AutoModelForSequenceClassification — task head included
clf_model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased-finetuned-sst-2-english')
with torch.no_grad():
    clf_out = clf_model(**encoded)
logits = clf_out.logits
print('\nClassification logits:', logits.shape)
print('Predicted classes:    ', logits.argmax(dim=-1).tolist())
print('Labels:               ', clf_model.config.id2label)

## 4. Saving and Loading Models Locally

In [ ]:
import os

SAVE_DIR = '/tmp/my_distilbert'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save
clf_model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print('Saved to:', SAVE_DIR)
print('Files:', os.listdir(SAVE_DIR))

# Reload
reloaded = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
reloaded_tok = AutoTokenizer.from_pretrained(SAVE_DIR)
tst = reloaded_tok('Transformers are amazing!', return_tensors='pt')
with torch.no_grad():
    logits2 = reloaded(**tst).logits
label = reloaded.config.id2label[logits2.argmax().item()]
print(f'\nReloaded model says: {label}')

## 5. Model Size Comparison

In [ ]:
import matplotlib.pyplot as plt

models_info = [
    ('DistilBERT-base', 66),
    ('BERT-base', 110),
    ('BERT-large', 340),
    ('GPT-2 small', 117),
    ('GPT-2 medium', 345),
    ('T5-small', 60),
    ('T5-base', 220),
    ('RoBERTa-base', 125),
]
names, params = zip(*models_info)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(names, params, color='steelblue')
ax.set_xlabel('Parameters (millions)')
ax.set_title('Model Size Comparison (Parameter Count)')
for bar, p in zip(bars, params):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            f'{p}M', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## Practice Exercises

**Exercise 1 — Zero-Shot Classification**
Use `pipeline('zero-shot-classification', model='facebook/bart-large-mnli')` to classify arbitrary texts into custom categories without any fine-tuning. Test with 5 news headlines and labels like `['politics', 'technology', 'sports', 'science']`.

**Exercise 2 — Subword Tokenisation Explorer**
Load tokenizers from `bert-base-uncased` (WordPiece), `roberta-base` (BPE), and `t5-small` (SentencePiece). Tokenise the same 5 sentences with each. Compare vocabulary sizes, number of tokens produced, and how rare words are handled differently.

**Exercise 3 — Feature Extraction Pipeline**
Use `pipeline('feature-extraction', model='distilbert-base-uncased')` to compute sentence embeddings. Embed 10 sentences from different topics, compute pairwise cosine similarity, and display a similarity heatmap. Verify that topically similar sentences cluster together.